In [ ]:
# excel 前兩行數據
# MallCode,	OR_Landmark_ID,	MallNameTc,	MallNameEn,	Id,	OR_POI_ID,	ReferenceId,	MallId,	Name_en,	Name_tc,	Name_sc,	CategoryNameEn,	ShopNumber
# tmtplaza,	42,	屯門市廣場,	tmtplaza,	1592,	803423,	540,	1,	&you,	&you,	&you,	Western Cuisine,	3212
# tmtplaza,	42,	屯門市廣場,	tmtplaza,	1605,	794645,	544,	1,	A-1 Bakery Premium,	A-1 Bakery Premium,	A-1 Bakery Premium,	Confectionery & Deli,	UG046

# 我的需求如下
# 1. 讀取Sino POI full list.xlsx的sheet Sino POIs，根據每個landmark id 分組，統計每個landmark id 下的相應poi的以下數據：
# No. of POIs - Booking	
# No. of Booking	
# No. of POIs - Voucher	
# No. of Voucher Transaction	
# No. of POIs - Takeaway	
# No. of Takeaway Transaction	
# Retail POIs Video Views
# 代碼請參考 Sino_poi.sql(請注意，代碼是基於landmark所有商戶統計的，我要的是在excel裏landmark含有的poi：指定商戶)
# 請修改代碼，讓他計算excel的poi的統計數據，而且我需要一次性呈現所有結果（像sql一樣），df只能有一個結果，不方便。
# 請保持SET @landmarkId = 37 的設定，我可以自己輸入landmark id生成結果。


In [ ]:
# MallCode,	OR_Landmark_ID,	MallNameTc,	MallNameEn,	Id,	OR_POI_ID,	ReferenceId,	MallId,	Name_en,	Name_tc,	Name_sc,	CategoryNameEn,	ShopNumber
# wanchai	N/A	灣仔	Wan Chai	1938	756721		12	Sugidama (The Hennessy)	杉玉居酒屋 (The Hennessy)	杉玉居酒屋 (The Hennessy)	Asian Cuisine	B&C
# wanchai	N/A	灣仔	Wan Chai	1943	31166		12	Wooloomooloo Steakhouse (The Hennessy)	Wooloomooloo Steakhouse (The Hennessy)	Wooloomooloo Steakhouse (The Hennessy)	Western Cuisine	1


In [ ]:
import pandas as pd
import pymssql

DB_info = {'server': '192.168.61.119:7622', 'user': 'BAReporting', 'password': 'KeHeCReme8he'}

landmark_id = 42
manual_or_poi_ids = []    # LandmarkId = N/A: [756721, 31166]
start_date = '2026-09-01'
end_date = '2026-10-01'

poi_source = pd.read_excel('Sino POI full list.xlsx', sheet_name='Sino POIs')
poi_data = (
    poi_source.rename(columns={
        'OR Landmark ID': 'LandmarkId',
        'OR POI ID': 'ORPoiId',
        'Id': 'MarsPoiId',
    })
    .assign(
        LandmarkId=lambda data: pd.to_numeric(data['LandmarkId'], errors='coerce'),
        ORPoiId=lambda data: pd.to_numeric(data['ORPoiId'], errors='coerce'),
        MarsPoiId=lambda data: pd.to_numeric(data['MarsPoiId'], errors='coerce'),
    )
)

landmark_pois = (
    poi_data.dropna(subset=['LandmarkId'])
    .dropna(subset=['ORPoiId', 'MarsPoiId'], how='all')
    .astype({'LandmarkId': 'int64', 'ORPoiId': 'Int64', 'MarsPoiId': 'Int64'})
    .loc[lambda data: data['LandmarkId'].eq(landmark_id), ['LandmarkId', 'ORPoiId', 'MarsPoiId']]
)
manual_pois = pd.DataFrame({
    'LandmarkId': landmark_id,
    'ORPoiId': manual_or_poi_ids,
    'MarsPoiId': pd.NA,
}).astype({'LandmarkId': 'int64', 'ORPoiId': 'Int64', 'MarsPoiId': 'Int64'})

selected_pois = (manual_pois if manual_or_poi_ids else landmark_pois).drop_duplicates()

if selected_pois.empty:
    raise ValueError(f'No POIs selected for landmark_id={landmark_id}.')

selected_pois

,LandmarkId,ORPoiId,MarsPoiId
0,42,803423,1592
1,42,794645,1605
2,42,9527,582
3,42,858431,2058
4,42,690055,2383
...,...,...,...
86,42,896415,2145
87,42,1006477,1316
88,42,864137,2075
89,42,446095,591


In [10]:
excel_poi_values = ', '.join(
    f"({int(or_poi_id) if pd.notna(or_poi_id) else 'NULL'}, "
    f"{int(mars_poi_id) if pd.notna(mars_poi_id) else 'NULL'})"
    for or_poi_id, mars_poi_id in selected_pois[['ORPoiId', 'MarsPoiId']].itertuples(index=False)
)

sql = f'''
SET NOCOUNT ON;

DECLARE @START datetime = %s;
DECLARE @END datetime = %s;

CREATE TABLE #ExcelPoi (
    ORPoiId int NULL,
    MarsPoiId int NULL
);

INSERT INTO #ExcelPoi (ORPoiId, MarsPoiId)
VALUES {excel_poi_values};

SELECT DISTINCT
    mars_poi.PoiId,
    mars_poi.ORPoiId
INTO #BboPoi
FROM mars.dbo.Poi AS mars_poi
INNER JOIN #ExcelPoi AS excel_poi
    ON (excel_poi.ORPoiId = mars_poi.ORPoiId AND mars_poi.RegionId = 0)
    OR (excel_poi.ORPoiId IS NULL AND excel_poi.MarsPoiId = mars_poi.PoiId);

DECLARE @MIN_TakeAwayOrderID int;

SELECT @MIN_TakeAwayOrderID = MAX(TakeAwayOrderID)
FROM mars.dbo.TakeAwayOrder WITH (NOLOCK)
WHERE CreateTime < @START;

SELECT
    [No. of POIs - Booking] = (
        SELECT COUNT(DISTINCT booking.PoiId)
        FROM mars.dbo.Booking AS booking
        INNER JOIN #BboPoi AS bbo_poi ON bbo_poi.PoiId = booking.PoiId
        WHERE booking.BookingTime >= @START
          AND booking.BookingTime < @END
          AND booking.[Status] = 10
    ),
    [No. of Booking] = (
        SELECT COUNT_BIG(*)
        FROM mars.dbo.Booking AS booking
        INNER JOIN #BboPoi AS bbo_poi ON bbo_poi.PoiId = booking.PoiId
        WHERE booking.BookingTime >= @START
          AND booking.BookingTime < @END
          AND booking.[Status] = 10
    ),
    [No. of POIs - Voucher] = (
        SELECT COUNT(DISTINCT service.PoiId)
        FROM mars.dbo.BizService AS service
        INNER JOIN #BboPoi AS bbo_poi ON bbo_poi.PoiId = service.PoiId
        WHERE service.ServiceStartTime < @END
          AND service.ServiceEndTime >= @END
          AND service.ServiceTypeId = 5
    ),
    [No. of Voucher Transaction] = (
        SELECT COUNT_BIG(*)
        FROM mars.dbo.OfferWallet AS wallet
        INNER JOIN mars.dbo.OfferPoi AS offer_poi ON offer_poi.OfferId = wallet.OfferId
        INNER JOIN mars.dbo.Offer AS offer ON offer.OfferId = wallet.OfferId
        INNER JOIN #BboPoi AS issue_poi ON issue_poi.PoiId = offer_poi.PoiId
        INNER JOIN #BboPoi AS redeem_poi ON redeem_poi.PoiId = wallet.RedeemPoiId
        WHERE wallet.CreateTime >= @START
          AND wallet.CreateTime < @END
          AND wallet.[Status] = 15
          AND offer.OfferType IN (7, 18, 22)
    ),
    [No. of POIs - Takeaway] = (
        SELECT COUNT(DISTINCT takeaway.PoiId)
        FROM mars.dbo.TakeAwayOrder AS takeaway WITH (NOLOCK)
        INNER JOIN #BboPoi AS bbo_poi ON bbo_poi.PoiId = takeaway.PoiId
        WHERE takeaway.TakeAwayOrderId >= @MIN_TakeAwayOrderID
          AND takeaway.CreateTime < @END
          AND takeaway.[Status] = 10
    ),
    [No. of Takeaway Transaction] = (
        SELECT COUNT_BIG(*)
        FROM mars.dbo.TakeAwayOrder AS takeaway WITH (NOLOCK)
        INNER JOIN #BboPoi AS bbo_poi ON bbo_poi.PoiId = takeaway.PoiId
        WHERE takeaway.TakeAwayOrderId >= @MIN_TakeAwayOrderID
          AND takeaway.CreateTime < @END
          AND takeaway.[Status] = 10
    ),
    [Retail POIs Video Views] = COALESCE((
        SELECT SUM(media.HitCount)
        FROM openrice3.dbo.Media AS media
        INNER JOIN #BboPoi AS bbo_poi ON bbo_poi.ORPoiId = media.PoiId
        INNER JOIN openrice3.dbo.Poi AS poi ON poi.PoiId = media.PoiId
        WHERE poi.PoiTypeId = 10
    ), 0);
'''

with pymssql.connect(**DB_info) as conn:
    report = pd.read_sql(sql, conn, params=(start_date, end_date))

report

C:\Users\lenalee\AppData\Local\Temp\ipykernel_80388\3599927889.py:99: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  report = pd.read_sql(sql, conn, params=(start_date, end_date))


,No. of POIs - Booking,No. of Booking,No. of POIs - Voucher,No. of Voucher Transaction,No. of POIs - Takeaway,No. of Takeaway Transaction,Retail POIs Video Views
0,3,189,8,0,0,0,0
